<a href="https://colab.research.google.com/github/Maxxx-VS/IMA_SibADI/blob/main/ML_2_1_%D0%9E%D0%B1%D1%83%D1%87%D0%B5%D0%BD%D0%B8%D0%B5_%D0%BD%D0%B5%D0%B9%D1%80%D0%BE%D1%81%D0%B5%D1%82%D0%B8_%D1%81_1_%D0%B2%D1%85%D0%BE%D0%B4%D0%BE%D0%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задание 2.1. Обучение нейросети с 1 входом (линейный нейрон)

In [1]:
!pip install -q onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 39.4 MB/s eta 0:00:00


In [4]:
import numpy as np
import pandas as pd
df = pd.read_csv('/content/multiregress-092022.csv')
df = df.drop('i', axis=1)
x = df.Me.to_numpy()
y = df.tc.to_numpy()
df.head(3)

,ne,Me,tc,ge,CO2,NOx
0,1200,140,80.7,233.16,5.88,665.25
1,1200,210,82.8,225.23,8.67,900.50
2,1200,252,83.2,232.83,10.58,957.19


In [5]:
import plotly.graph_objects as go
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=x.flatten(), y=y.flatten(),
                          mode='markers', name='y',
                          marker=dict(color='black', size=7, opacity=0.8)))
fig1.update_layout(title_text="y(x)", title_font_size=20,
                   xaxis_title="x", yaxis_title="y")
fig1.show()

In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=True, with_std=True)
x_s = scaler.fit_transform(x.reshape(-1, 1))
x_s = x_s.reshape(22,)

In [7]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=x_s.flatten(), y=y.flatten(),
                          mode='markers', name='y',
                          marker=dict(color='black', size=7, opacity=0.8)))
fig2.update_layout(title_text="Стандартизованные по х данные", title_font_size=20,
                   xaxis_title="x_s", yaxis_title="y")
fig2.show()

In [8]:
np.random.seed(42)
b = np.random.rand(1)
w = np.random.randn(1)

In [9]:
n_epochs = 500
losses = []

lr = 0.01
for epoch in range(n_epochs):
    yhat = b + w * x_s
    error = yhat - y
    loss = np.mean(error**2)
    losses.append(loss)

    b_grad = 2 * error.mean()
    w_grad = 2 * (x_s * error).mean()

    if epoch % 50 == 0:
        print('Epoch: ', epoch, ' b_grad: ', b_grad, ' w_grad: ', w_grad, ' b: ', b, ' w: ', w)

    b -= lr * b_grad
    w -= lr * w_grad

print('Обучение закончено: ', ' b: ', b, ' w: ', w)


Epoch:  0  b_grad:  -163.03273794412345  w_grad:  -5.0632939034420295  b:  [0.37454012]  w:  [-1.11188012]
Epoch:  50  b_grad:  -59.37158002083823  w_grad:  -1.843898121003537  b:  [52.20511908]  w:  [0.49781777]
Epoch:  100  b_grad:  -21.621329302455315  w_grad:  -0.671491788839099  b:  [71.08024444]  w:  [1.08402094]
Epoch:  150  b_grad:  -7.873832575133411  w_grad:  -0.24453694992265476  b:  [77.9539928]  w:  [1.29749836]
Epoch:  200  b_grad:  -2.8674110899458496  w_grad:  -0.08905294282281319  b:  [80.45720355]  w:  [1.37524036]
Epoch:  250  b_grad:  -1.0442241793038185  w_grad:  -0.03243038169859392  b:  [81.368797]  w:  [1.40355164]
Epoch:  300  b_grad:  -0.38027478531631687  w_grad:  -0.011810161728280543  b:  [81.7007717]  w:  [1.41386175]
Epoch:  350  b_grad:  -0.13848454691381432  w_grad:  -0.004300902818368712  b:  [81.82166682]  w:  [1.41761638]
Epoch:  400  b_grad:  -0.05043187314660971  w_grad:  -0.0015662584034459254  b:  [81.86569315]  w:  [1.4189837]
Epoch:  450  b_gra

In [10]:
fig3 = go.Figure()
fig3.add_trace(go.Scatter(y=losses,
                          mode='markers', name='loss',
                          marker=dict(color='black', size=7), opacity=0.8))
fig3.update_layout(title_text="MSE vs epoch", title_font_size=20,
                   xaxis_title="epoch", yaxis_title="MSE")
fig3.show()

In [11]:
fig2.add_trace(go.Scatter(x=x_s.flatten(), y=yhat.flatten(),
                          mode='markers+lines', name='yhat',
                          marker=dict(color='green', size=7), opacity=0.8))
fig2.show()

In [12]:
import torch
from torch import nn
model = nn.Linear(bias=True, in_features=1, out_features=1)

In [13]:
x_torch = torch.as_tensor(x_s.reshape([-1, 1])).float()
y_torch = torch.as_tensor(y.reshape([-1, 1])).float()

In [14]:
from torch import optim
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [15]:
n_epochs = 500
losses = []

for epoch in range(n_epochs):
    loss = 0.0
    model.train()
    optimizer.zero_grad()
    y_hat_torch = model(x_torch)
    loss = loss_fn(y_hat_torch, y_torch)
    loss.backward()
    optimizer.step()
    losses.append(loss.detach().numpy())
    if epoch % 50 == 0:
        print('Epoch: {}, Loss: {:.2f}'.format(epoch, loss))

Epoch: 0, Loss: 6803.21
Epoch: 50, Loss: 902.57
Epoch: 100, Loss: 120.02
Epoch: 150, Loss: 16.24
Epoch: 200, Loss: 2.48
Epoch: 250, Loss: 0.66
Epoch: 300, Loss: 0.41
Epoch: 350, Loss: 0.38
Epoch: 400, Loss: 0.38
Epoch: 450, Loss: 0.38


In [16]:
print('Смещение ', model.bias)
print('Вес ', model.weight)

Смещение  Parameter containing:
tensor([81.8875], requires_grad=True)
Вес  Parameter containing:
tensor([[1.4197]], requires_grad=True)


In [17]:
fig3.add_trace(go.Scatter(y=losses,
                          mode='markers+lines', name='PyTorch loss',
                          marker=dict(color='red', size=3), opacity=0.8))
fig3.show()

In [18]:
torch.onnx.export(model, x_torch[0], "neuron_one_input.onnx",
                  input_names=['x'], output_names=['y'], dynamo=False)

/tmp/ipykernel_574/23149300.py:1: DeprecationWarning:

You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html

